In [1]:
import pyEDM
import pandas as pd
import numpy as np
import random
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [2]:
# top 20 features from paper + target variable
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    "Solidity", "texture_uniformity", "texture_smoothness",
    "texture_average_gray_level", "texture_entropy",
    "texture_average_contrast", "H90", "H180", "Hflip",
    "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")
df = df.asfreq("D")

df_filled = df.copy()

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    df_norm[col] = (df_filled[col] - mu) / sigma

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
df_mv = df_mv[["t"] + features]

target = "Lpoly_expected_ml"
predictors = [col for col in features if col != target]

df_mv.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,texture_average_gray_level,texture_entropy,texture_average_contrast,H90,H180,Hflip,Extent,EquivDiameter,ConvexArea,ConvexPerimeter
0,1,0.285287,1.059338,0.967595,0.956625,1.139942,1.226418,0.842266,-1.228078,-0.191169,...,-1.183856,0.480870,-1.005885,-0.901014,0.861091,1.128490,0.102206,1.092577,1.111408,1.118301
1,2,0.065656,1.016480,0.917837,0.908034,1.109145,1.007938,0.563761,-1.247322,0.526462,...,-1.141946,0.640061,-0.878977,-1.001016,0.340277,0.680457,0.656972,1.062576,0.999125,1.020599
2,3,-0.128818,0.671753,0.512658,0.637894,0.852846,0.719154,0.645106,-1.070290,0.486697,...,-1.073251,0.996070,-0.759717,-0.976040,0.078281,0.314316,0.520649,0.807369,0.659696,0.771253
3,4,-0.018094,0.379651,0.165389,0.433051,0.632838,0.501162,0.894757,-0.832234,0.370645,...,-1.029278,1.106370,-0.779254,-0.852571,0.097279,0.420381,0.192816,0.595165,0.375226,0.570793
4,5,0.891676,0.756008,0.593485,0.724495,0.928235,0.917121,0.418304,-1.071257,0.091247,...,-1.184920,0.232139,-0.948879,-0.995896,0.312944,0.654387,0.256093,0.879835,0.773202,0.863842


In [3]:
env = pd.read_csv("../data/environment_all.csv")

# could be relevant but too many missing values for ema imputation to be effective
env = env.drop(columns=[
    'fluorescent_dissolved_organic_matter_eco',
    'sea_water_turbidity_eco',
    'waterlevel_predicted_m',
    'mass_concentration_of_oxygen_in_sea_water_seaphox',
    'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox',
    'fractional_saturation_of_oxygen_in_sea_water_seaphox',
    'sea_water_ph_reported_on_total_scale_seaphox_external'
])

env["date"] = pd.to_datetime(env["date"].astype(str), format="%Y%m%d")
env = env.sort_values("date").set_index("date")
env = env.asfreq("D")

env_features = env.columns.tolist()
env_filled = env.copy()

for col in env_features:
    ema = env[col].ewm(span=30, adjust=False).mean()
    env_filled[col] = env[col].fillna(ema)

df_norm = env_filled.copy()

for col in env_features:
    mu = env_filled[col].mean()
    sigma = env_filled[col].std()
    df_norm[col] = (env_filled[col] - mu) / sigma

df_env = df_norm.copy()
df_env = df_env.reset_index()
df_env["t"] = np.arange(1, len(df_env) + 1)
df_env = df_env[["t"] + env_features]

df_all = pd.merge(df_mv, df_env, on="t", how="inner")

df_env = df_env.merge(df_mv[["t", target]], on="t", how="inner")
df_env.head()

,t,mass_concentration_of_chlorophyll_in_sea_water_ctd,sea_water_electrical_conductivity_ctd,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m,Lpoly_expected_ml
0,1,0.386650,1.505381,0.361222,-0.887083,0.781443,1.526531,-0.299634,0.288179,-0.014762,1.008903,-1.294158,1.006837,0.285287
1,2,0.718488,1.473634,0.375660,-0.833821,0.519131,1.488148,-0.107828,0.303818,-0.095281,1.224096,-0.784883,0.654760,0.065656
2,3,1.065658,0.860840,0.326331,-0.349615,0.544113,0.835239,-0.115066,0.714121,-0.069400,1.365032,-0.344831,0.304790,-0.128818
3,4,1.402899,0.352516,0.254143,0.008697,0.331765,0.303019,-0.799053,-0.682379,-0.785444,1.101345,-0.532718,0.348009,-0.018094
4,5,1.678048,0.440744,0.238502,-0.079670,0.181872,0.407880,-0.665150,0.782198,-0.788319,0.808864,-0.370789,0.219406,0.891676


In [4]:
df_all.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m
0,1,0.285287,1.059338,0.967595,0.956625,1.139942,1.226418,0.842266,-1.228078,-0.191169,...,0.361222,-0.887083,0.781443,1.526531,-0.299634,0.288179,-0.014762,1.008903,-1.294158,1.006837
1,2,0.065656,1.016480,0.917837,0.908034,1.109145,1.007938,0.563761,-1.247322,0.526462,...,0.375660,-0.833821,0.519131,1.488148,-0.107828,0.303818,-0.095281,1.224096,-0.784883,0.654760
2,3,-0.128818,0.671753,0.512658,0.637894,0.852846,0.719154,0.645106,-1.070290,0.486697,...,0.326331,-0.349615,0.544113,0.835239,-0.115066,0.714121,-0.069400,1.365032,-0.344831,0.304790
3,4,-0.018094,0.379651,0.165389,0.433051,0.632838,0.501162,0.894757,-0.832234,0.370645,...,0.254143,0.008697,0.331765,0.303019,-0.799053,-0.682379,-0.785444,1.101345,-0.532718,0.348009
4,5,0.891676,0.756008,0.593485,0.724495,0.928235,0.917121,0.418304,-1.071257,0.091247,...,0.238502,-0.079670,0.181872,0.407880,-0.665150,0.782198,-0.788319,0.808864,-0.370789,0.219406


https://sugiharalab.github.io/EDM_Documentation/parameters/

In [5]:
def one_simplex(df, target, features, E=4, Tp=1):
    # Randomly select 3 features (+ the target) for the simplex projection
    chosen_features = random.sample(features, 3)
    columns = [target] + chosen_features
    columns_str = " ".join(columns) # has to be 'space separated' idk ????

    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        columns=columns_str,
        target=target,
        E=E,
        tau=1,
        Tp=Tp,
        lib=f"1 {N}",
        pred=f"1 {N}"
    )

    obs = res["Observations"].to_numpy()
    pred = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]

    if len(obs) < 10 or np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan, chosen_features

    rho = np.corrcoef(obs, pred)[0, 1]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mae = np.mean(np.abs(obs - pred))
    return rho, rmse, mae, chosen_features

def multiview_big(df, target, features, Tp, n_trials=500):
    results = []

    for i in range(n_trials):
        rho, rmse, mae, chosen = one_simplex(df, target, features, E=4, Tp=Tp)

        results.append({
            "rho": rho,
            "rmse": rmse,
            "mae": mae,
            "features": chosen
        })

    return pd.DataFrame(results)

def multiview_yes(df_mv, target, predictors):
    # wrapper main function
    x = df_mv[target].to_numpy()

    summary_rows = []
    feature_importance_by_tp = {}
    
    for Tp in range(1, 32):
        mv = multiview_big(df_mv, target, predictors, Tp, n_trials=500)

        acf = abs(pd.Series(x).autocorr(lag=Tp))
        mv["rho_eff"] = mv["rho"] - acf

        # summary stats
        summary_rows.append({
            "Tp": Tp,
            "rho_eff_mean": mv["rho_eff"].mean(),
            "rmse_mean": mv["rmse"].mean(),
            "mae_mean": mv["mae"].mean(),
            "n_models": len(mv)
        })

        # feature importance (top 5%)
        top = mv.nlargest(int(0.05 * len(mv)), "rho_eff")

        counts = Counter()
        for feats in top["features"]:
            for f in feats:
                counts[f] += 1

        importance = pd.Series(counts) / len(top)
        feature_importance_by_tp[Tp] = importance.sort_values(ascending=False)
    
    return summary_rows, feature_importance_by_tp

In [6]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_mv, target, predictors)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.189670   0.440394  0.084095       500
1    2      0.442064   0.402713  0.079076       500
2    3      0.584680   0.377887  0.080149       500
3    4      0.271022   0.810168  0.181730       500
4    5      0.125453   0.945559  0.217064       500
5    6      0.165731   0.965579  0.225987       500
6    7      0.148857   0.979679  0.238021       500
7    8      0.080950   0.999289  0.240920       500
8    9      0.099523   0.999263  0.241141       500
9   10      0.077811   0.997040  0.243595       500
10  11      0.081395   1.000141  0.245698       500
11  12      0.058339   1.024149  0.264741       500
12  13      0.029618   1.037618  0.277121       500
13  14      0.055383   1.049676  0.282777       500
14  15      0.093174   1.047216  0.288372       500
15  16      0.133119   1.032154  0.294482       500
16  17      0.173536   1.007280  0.291643       500
17  18      0.260690   0.974868  0.294040       500
18  19      

In [7]:
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)

importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
print(importance_all)

     Tp             Feature  Proportion
0     1            Solidity        0.52
1     1          ConvexArea        0.44
2     1              Extent        0.36
3     1         Orientation        0.32
4     1                Area        0.28
..   ..                 ...         ...
472  31     MinorAxisLength        0.12
473  31        Eccentricity        0.12
474  31  texture_smoothness        0.08
475  31                 H90        0.08
476  31           Biovolume        0.08

[477 rows x 3 columns]


In [8]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_env, target, env_features)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.179949   0.458060  0.118622       500
1    2      0.434510   0.421959  0.114013       500
2    3      0.575071   0.403981  0.117833       500
3    4      0.269652   0.809769  0.203743       500
4    5      0.127998   0.934034  0.233130       500
5    6      0.189171   0.943038  0.240008       500
6    7      0.192750   0.956040  0.249813       500
7    8      0.113551   0.993351  0.262246       500
8    9      0.132951   0.982004  0.260908       500
9   10      0.109560   0.988277  0.268506       500
10  11      0.147484   0.986654  0.275386       500
11  12      0.193983   0.991191  0.289667       500
12  13      0.228020   0.971668  0.291066       500
13  14      0.270793   0.971626  0.289601       500
14  15      0.266164   0.989282  0.296030       500
15  16      0.235900   0.996419  0.299905       500
16  17      0.248361   0.968334  0.292483       500
17  18      0.217199   0.993466  0.295969       500
18  19      

In [9]:
importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
print(importance_all)

     Tp                                            Feature  Proportion
0     1                              sea_water_sigma_t_ctd        0.84
1     1              sea_water_electrical_conductivity_ctd        0.68
2     1                          sea_water_temperature_ctd        0.52
3     1                   sea_water_practical_salinity_ctd        0.44
4     1                                         air_temp_c        0.28
..   ..                                                ...         ...
241  31              sea_water_electrical_conductivity_ctd        0.60
242  31                              sea_water_sigma_t_ctd        0.52
243  31  mass_concentration_of_chlorophyll_in_sea_water...        0.52
244  31                   sea_water_practical_salinity_ctd        0.48
245  31                          sea_water_temperature_ctd        0.12

[246 rows x 3 columns]


In [10]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_all, target, predictors+env_features)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.186302   0.446444  0.094703       500
1    2      0.439084   0.409982  0.090092       500
2    3      0.580646   0.388502  0.092244       500
3    4      0.265378   0.814437  0.188421       500
4    5      0.124368   0.943840  0.220447       500
5    6      0.176930   0.957095  0.227952       500
6    7      0.175251   0.966530  0.235885       500
7    8      0.104366   0.990845  0.241444       500
8    9      0.127685   0.984279  0.239687       500
9   10      0.098510   0.986435  0.242043       500
10  11      0.111238   0.991121  0.248666       500
11  12      0.117709   1.005525  0.265066       500
12  13      0.114583   1.007936  0.274552       500
13  14      0.157261   1.013390  0.277733       500
14  15      0.197462   1.013335  0.283642       500
15  16      0.208958   1.008514  0.290659       500
16  17      0.216009   0.997079  0.291752       500
17  18      0.231657   0.997632  0.296345       500
18  19      

In [ ]:
#rho_eff over Tp
fig = px.line(summary_df, x="Tp", y="rho_eff_mean", title="Effective Forecast Skill over Prediction Interval (Tp)", markers=True)
fig.update_layout(xaxis_title="Prediction Interval (Tp)", yaxis_title="Effective Forecast Skill (rho_eff)")
fig.show()

In [29]:
#classic RMSE & MAE over Tp

fig = go.Figure()
fig.add_scatter(x=summary_df["Tp"], y=summary_df["rmse_mean"], name="RMSE", mode="lines+markers")
fig.add_scatter(x=summary_df["Tp"], y=summary_df["mae_mean"],  name="MAE",  mode="lines+markers")
fig.update_layout(title="RMSE & MAE over Prediction Interval (Tp)", xaxis_title="Prediction Interval (Tp)", yaxis_title="Errors")
fig.show()


In [27]:
#heatmap of feat importance by Tp
pivot = importance_all.pivot_table(index="Feature", columns="Tp", values="Proportion", aggfunc="mean")
px.imshow(pivot, aspect="auto", color_continuous_scale="Viridis", title="Feature Importance by Prediction Interval (Tp)",labels={"x": "Prediction Interval (Tp)"}).show()

In [32]:
#features importance bar chart
mean_imp = importance_all.groupby("Feature")["Proportion"].mean().sort_values().tail(15)
px.bar(mean_imp, orientation="h", title="Mean Feature Importance",
       labels={"value": "Proportion"}).update_layout(showlegend=False).show()